# Phase 4: Multi-View Stereo (MVS) — Dense Depth Estimation

**Pipeline position:**
```
Phase 1: Synthetic Scene        ✓
Phase 2: Feature Matching       ✓
Phase 3: Structure from Motion  ✓
Phase 4: Dense Depth (MVS)      ← YOU ARE HERE
Phase 5: Mesh Reconstruction
...
```

---

## Sparse vs. Dense: Why We Need MVS

Phase 3 gave us a **sparse** point cloud — a few hundred to thousand points, only at feature locations (building corners, texture edges).  This is not enough to:
- Reconstruct a watertight mesh
- Capture building faces and ground surfaces
- Produce a texture-able 3-D model

**Multi-View Stereo (MVS)** estimates depth for **every pixel** in every image, producing a **dense** point cloud with millions of points.  The key idea: use multiple images of the same scene from nearby viewpoints to solve the correspondence problem pixel-by-pixel.

---

## The Stereo Geometry

### Epipolar Constraint

Consider two cameras observing the same scene point $\mathbf{X}$.  The **epipolar constraint** says: the projection of $\mathbf{X}$ in the right image must lie on a specific line (the **epipolar line**) determined by the left image pixel and the relative camera geometry.

This reduces a 2-D search (find the matching pixel anywhere in the right image) to a **1-D search** (search along the epipolar line).

### Stereo Rectification

For arbitrary camera orientations, epipolar lines are diagonal — inconvenient.  **Rectification** warps both images so that epipolar lines become horizontal rows.  After rectification:

- Matching pixel in right image is found on the **same row**
- The match is at a horizontal offset $d$ (disparity)

```
Left image:   ●  ← building corner at pixel (u, v)
              ↓ epipolar line is now horizontal
Right image:  ←●  same row v, shifted left by d pixels
```

### Disparity to Depth

Once we have disparity $d$ (pixels), depth follows directly:

$$Z = \frac{f \cdot B}{d}$$

where:
- $f$ = focal length in pixels
- $B$ = baseline (distance between camera centres, in metres)
- $d$ = disparity in pixels

**Intuition:** a point far away shifts very little between left and right images (small $d$ → large $Z$).  A close point shifts a lot (large $d$ → small $Z$).  This is exactly how human binocular vision estimates depth.

---

## The SGBM Algorithm

We use **Semi-Global Block Matching (SGBM)** — a major improvement over naive block matching.

### Naive Block Matching
For each pixel, compare a small patch around it in the left image to patches at all disparity levels in the right image.  Pick the disparity with the lowest matching cost.  **Problem:** flat, textureless regions have no distinctive patches — the matching cost is nearly the same for all disparities.

### Semi-Global Matching (Hirschmuller, 2008)
Add a **smoothness penalty** along multiple 1-D paths through the image.  The disparity must not change abruptly unless the matching cost strongly justifies it:

$$E(d) = \underbrace{C(p, d)}_{\text{matching cost}} + \underbrace{P_1 \cdot \mathbf{1}[|d - d'| = 1]}_{\text{small step penalty}} + \underbrace{P_2 \cdot \mathbf{1}[|d - d'| > 1]}_{\text{large step penalty}}$$

This produces **smooth depth maps** with sharp discontinuities at object boundaries — much more suitable for surface reconstruction than naive block matching.

---

## Multi-View Fusion

A single stereo pair has limitations:
- **Blind spots**: regions visible in one image but occluded in the other
- **Low-texture ambiguity**: flat surfaces produce uncertain disparity
- **Noise**: SGBM errors are independent per pair

By computing depth from **multiple neighbouring pairs** and averaging, we:
- Fill occlusion holes (covered by another pair)
- Reduce noise (uncorrelated errors cancel)
- Increase coverage (each pixel covered by several estimates)

In [4]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import cv2
import open3d as o3d
import matplotlib.pyplot as plt
import json
%matplotlib inline

In [5]:
img_dir    = os.path.join('..', 'data', 'synthetic', 'images')
poses_path = os.path.join('..', 'data', 'synthetic', 'GT_poses.json')
depth_dir  = os.path.join('..', 'data', 'synthetic', 'GT_depth')
assert os.path.isdir(img_dir), "Run phase1_data_generation.ipynb first."

img_files = sorted(os.listdir(img_dir))
images    = [cv2.imread(os.path.join(img_dir, f)) for f in img_files]

with open(poses_path) as f:
    gt_poses = json.load(f)

K = np.array(gt_poses[0]['K'])
print(f"Loaded {len(images)} images, focal = {K[0,0]:.1f} px")

Loaded 16 images, focal = 692.8 px


---

## Why Phase 4 Uses Ground Truth Poses (Not Phase 3 Output)

You might expect this phase to load the **estimated** camera poses from Phase 3.  It does not — this is an intentional pedagogical choice, and understanding why is important.

### The Scale Ambiguity Problem

Phase 3 (SfM) recovers poses in **arbitrary scale**: the initial camera pair's baseline is fixed to $|\mathbf{t}| = 1$ by convention.  The actual physical baseline in metres is unknown from images alone.

Depth estimation uses:
$$Z = \frac{f \cdot B}{d}$$

If $B$ (baseline) is in SfM units instead of metres, then $Z$ comes out in SfM units — not metres.  The dense point cloud and mesh would be in unitless SfM scale, making Chamfer distance (Phase 8) meaningless.

### What a Real Pipeline Does to Recover Metric Scale

| Method | How |
|---|---|
| GPS-tagged images | Camera positions from onboard GPS → direct metric baseline |
| Ground Control Points | Surveyed markers at known world coordinates → Procrustes alignment |
| IMU integration | Inertial sensors integrate metric translations between frames |

### What This Notebook Does

We load **GT poses** (metric, from Phase 1) to produce metre-scale depth maps.  
This isolates MVS quality from SfM scale error so you can see SGBM's intrinsic accuracy.

**Phase 8** evaluates Phase 3's estimated poses in **degrees** (rotation error) — a scale-independent metric that can be computed without resolving scale.

---

## Step 1 — Single Stereo Pair: Rectify → SGBM → Depth

We walk through a single pair (image 0 and image 1) step by step to see the intermediate results.

**Expected outcome:** the depth map should show buildings as closer (smaller depth values) than the ground plane, since cameras are at 50 m altitude and buildings are 5–30 m tall → their tops are 20–45 m away.

In [6]:
from src.mvs.depth_estimator import compute_depth_map

i, j = 0, 1  # neighbouring cameras in the grid
R_i, t_i = np.array(gt_poses[i]['R']), np.array(gt_poses[i]['t'])
R_j, t_j = np.array(gt_poses[j]['R']), np.array(gt_poses[j]['t'])

# Compute stereo baseline
eye_i = np.array(gt_poses[i]['eye'])
eye_j = np.array(gt_poses[j]['eye'])
baseline = np.linalg.norm(eye_i - eye_j)
print(f"Camera pair ({i}, {j})")
print(f"Baseline = {baseline:.2f} m")
print(f"Expected depth range: {K[0,0] * baseline / 128:.1f} – {K[0,0] * baseline / 1:.0f} m")
print(f"  (Z = f*B/d,  d in [1, 128])")
print()

depth = compute_depth_map(images[i], images[j], K, R_i, t_i, R_j, t_j)
valid = depth[depth > 0]
print(f"Valid pixels : {(depth > 0).sum():,} / {depth.size:,}  ({100*(depth>0).mean():.1f}%)")
print(f"Depth range  : {valid.min():.1f} – {valid.max():.1f} m")

Camera pair (0, 1)
Baseline = 20.00 m
Expected depth range: 108.3 – 13856 m
  (Z = f*B/d,  d in [1, 128])



error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\imgwarp.cpp:1671: error: (-215:Assertion failed) ((map1.type() == CV_32FC2 || map1.type() == CV_16SC2) && map2.empty()) || (map1.type() == CV_32FC1 && map2.type() == CV_32FC1) in function 'cv::remap'


In [ ]:
gt_d = np.load(os.path.join(depth_dir, f'depth_{i:04d}.npy'))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB))
axes[0].set_title('RGB Image 0')
axes[0].axis('off')

clip = np.percentile(valid, 97) if len(valid) else 60
im1 = axes[1].imshow(depth, cmap='plasma_r', vmin=0, vmax=clip)
plt.colorbar(im1, ax=axes[1], label='Depth (m)')
axes[1].set_title('Estimated Depth (SGBM)\nbright = close, dark = far')
axes[1].axis('off')

gt_valid = gt_d[gt_d > 0]
im2 = axes[2].imshow(gt_d, cmap='plasma_r',
                     vmin=0,
                     vmax=np.percentile(gt_valid, 97) if len(gt_valid) else 60)
plt.colorbar(im2, ax=axes[2], label='Depth (m)')
axes[2].set_title('GT Depth (from Phase 1)\nfor comparison')
axes[2].axis('off')

plt.suptitle(f'Single Stereo Pair: Image {i} using Image {j} as reference', fontsize=12)
plt.tight_layout()
plt.show()

# Pixel-wise error where both maps have valid depth
common = (depth > 0) & (gt_d > 0)
if common.sum() > 0:
    err = np.abs(depth[common] - gt_d[common])
    print(f"Depth error vs GT (valid pixels only):")
    print(f"  Mean absolute error : {err.mean():.3f} m")
    print(f"  Median              : {np.median(err):.3f} m")
    print(f"  90th percentile     : {np.percentile(err, 90):.3f} m")

### Key Insight: Coverage vs. Accuracy Trade-off

Notice the **invalid pixels** (depth = 0) in the estimated map.  These occur where SGBM cannot find a reliable match:

| Region | Why invalid |
|---|---|
| Sky pixels | No surface — infinite depth |
| Occlusion boundaries | Left image sees it, right image doesn't |
| Textureless surfaces | Flat uniform colour — ambiguous disparity |

Increasing `num_disparities` extends the detectable range but slows SGBM.  Increasing `block_size` reduces noise but blurs depth edges.  Multi-view fusion addresses coverage; it cannot fix systematic SGBM errors.

---

## Step 2 — Multi-View Depth Estimation

For each image $i$, we compute a stereo depth map against each of its $k$ nearest neighbours, then fuse them by masked averaging (only include a depth estimate at pixel $(u,v)$ if it is valid and consistent across neighbours).

We use `neighbors=1` here for notebook speed.  In a production pipeline you would use `neighbors=2` or `neighbors=3` for better coverage.

In [ ]:
from src.mvs.depth_estimator import estimate_all_depths

# Use first 4 images for speed; remove [:4] for full pipeline
subset_images = images[:4]
subset_poses  = gt_poses[:4]

print("Estimating dense depth for 4 images (neighbors=1)...")
depth_maps = estimate_all_depths(subset_images, subset_poses, neighbors=1)

print("\nPer-image depth stats:")
for k, d in enumerate(depth_maps):
    v = d[d > 0]
    print(f"  Image {k}: {(d>0).mean()*100:.1f}% valid, "
          f"depth {v.min():.1f}–{v.max():.1f} m" if len(v) else f"  Image {k}: no valid depth")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for k in range(4):
    # Top row: RGB
    axes[0][k].imshow(cv2.cvtColor(subset_images[k], cv2.COLOR_BGR2RGB))
    axes[0][k].set_title(f'Image {k}', fontsize=9)
    axes[0][k].axis('off')

    # Bottom row: depth map
    d = depth_maps[k]
    v = d[d > 0]
    if len(v):
        im = axes[1][k].imshow(d, cmap='plasma_r',
                               vmin=0, vmax=np.percentile(v, 95))
        plt.colorbar(im, ax=axes[1][k], fraction=0.046, label='m')
    axes[1][k].set_title(f'Depth {k}: {(d>0).mean()*100:.0f}% valid', fontsize=9)
    axes[1][k].axis('off')

plt.suptitle('Multi-View Depth Maps (top: RGB, bottom: estimated depth)', fontsize=12)
plt.tight_layout()
plt.show()

---

## Step 3 — Depth Fusion: Depth Maps → Dense Point Cloud

We back-project every valid depth pixel into world space:

$$\mathbf{X}_{cam} = Z \cdot K^{-1} \begin{bmatrix} u \\ v \\ 1 \end{bmatrix}, \qquad \mathbf{X}_{world} = R^\top (\mathbf{X}_{cam} - \mathbf{t})$$

Each valid pixel becomes one 3-D point, coloured by the corresponding RGB value.  After back-projection from all images, we voxel-downsample to remove duplicates and reduce noise.

**Voxel downsampling:** divide 3-D space into a regular grid of voxels (size = `voxel_size` metres).  Replace all points inside each voxel with their centroid.  This removes duplicate observations and reduces point count while preserving coverage.

In [ ]:
from src.mvs.depth_fusion import fuse_depth_maps

print("Fusing depth maps into dense point cloud...")
pcd_dense = fuse_depth_maps(depth_maps, subset_images, subset_poses, voxel_size=0.5)

out_dir  = os.path.join('..', 'outputs')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'dense_cloud_subset.ply')
o3d.io.write_point_cloud(out_path, pcd_dense)

pts = np.asarray(pcd_dense.points)
print(f"Dense cloud: {len(pts):,} points")
print(f"Saved: {out_path}")
print(f"\nPoint cloud extents:")
print(f"  X: {pts[:,0].min():.1f} – {pts[:,0].max():.1f} m")
print(f"  Y: {pts[:,1].min():.1f} – {pts[:,1].max():.1f} m")
print(f"  Z: {pts[:,2].min():.1f} – {pts[:,2].max():.1f} m")

In [ ]:
pts    = np.asarray(pcd_dense.points)
colors = np.asarray(pcd_dense.colors)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top view (X-Y)
sc = axes[0].scatter(pts[:,0], pts[:,1], s=0.5,
                     c=colors if len(colors) else pts[:,2],
                     cmap=None if len(colors) else 'viridis', alpha=0.4)
axes[0].set_title(f'Dense Cloud — Top View  ({len(pts):,} pts)')
axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')
axes[0].set_aspect('equal')

# Side view (X-Z)
axes[1].scatter(pts[:,0], pts[:,2], s=0.5, c=pts[:,2], cmap='viridis', alpha=0.4)
axes[1].set_title('Dense Cloud — Side View')
axes[1].set_xlabel('X (m)'); axes[1].set_ylabel('Z / altitude (m)')

plt.suptitle('Dense Point Cloud from MVS Fusion (4-camera subset)', fontsize=12)
plt.tight_layout()
plt.show()

print("Compare to Phase 1 scene bounds: X=[0,100], Y=[0,100], Z=[0,35]")
print("The cloud covers a subset of the scene (only 4 of 16 cameras used here).")

---

## Summary

| Step | Details |
|---|---|
| Algorithm | Semi-Global Block Matching (SGBM) |
| Disparity levels | 128 (max detectable shift) |
| Block size | 7×7 px (matching window) |
| Fusion | Masked mean across neighbours |
| Downsampling | Voxel grid (0.5 m) |
| Output | Dense coloured point cloud (.ply) |

### Questions to Think About

- The formula $Z = fB/d$ shows depth is inversely proportional to disparity.  What is the minimum detectable depth?  The maximum?  How do they depend on the number of disparity levels?
- Why does SGBM fail on the sky?  What depth value would you assign to sky pixels, and why?
- Try increasing `neighbors` from 1 to 2.  How does coverage and accuracy change?
- The voxel size of 0.5 m determines point cloud density.  If a building wall is 10 m wide and 20 m tall, roughly how many points will represent it at 0.5 m resolution?

---

**Next → [Phase 5: Mesh Reconstruction](phase5_mesh.ipynb)**